In [1]:
# Importing Modules
import torch
import pandas as pd
import numpy as np
import itertools
from sklearn.model_selection import train_test_split
from src.dataloader import loadData
from src.model import SimpleGCN, SimpleGAT, SimpleGIN, SimpleGraphSAGE
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

In [2]:
# Loading ESOL data
esol_data = pd.read_csv("data/train/ESOL.csv")

In [3]:
# Models
model_dict = {"SimpleGCN":SimpleGAT, 
		"SimpleGAT":SimpleGAT, 
		"SimpleGIN":SimpleGIN,
		"SimpleGraphSAGE":SimpleGraphSAGE}

# Function to run analysis
def RunGNN(data, dataName, modelName, h_dim, b_size, lr):
    
	# SMILES (To be used to generate molecular graph, training and testing)
	smiles_X = data["smiles"]
	X = smiles_X.to_numpy()

	# Target labels (log solubility values)
	y = data["target"]
	y = y.to_numpy()

	# Train test split
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

	# Loading dataset
	train_loader = loadData(X_train, y_train, batch_size=b_size)
	test_loader = loadData(X_test, y_test, batch_size=b_size)

	# Model
	model = model_dict[modelName](num_features=11, hidden_dim=h_dim, num_classes=1)

	# Model training
	model = TrainGNN(model, train_loader, epochs=100, learning_rate=lr)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Outfile to write results
	out = open(f"results/GNN/output_{modelName}_{dataName}.txt", "a+")
	out.write(f'LR = {lr}\n')
	out.write(f'h_dim = {h_dim}\n')
	out.write(f'b_size = {b_size}\n')

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)
	out.write(f'MAE = {round(mae, 2)}\n')

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
	out.write(f'RMSE = {round(rmse, 2)}\n')

	r, p_val = pearsonr(y_test, y_pred)
	out.write(f'Pearson correlation coefficient = {round(r, 2)}, P-value= {round(p_val, 4)}\n')

	out.close()

In [4]:
# Hyperparameter dict
hyperparams = {
    'lr': [0.01, 0.005, 0.001],
    'h_dim': [16, 32, 64, 128],
    'b_size': [4, 8, 16, 32]}

# Grid search list
keys = hyperparams.keys()
values = hyperparams.values()
grid_search_list = [dict(zip(keys, combination)) for combination in itertools.product(*values)]

In [5]:
# Performing grid-search for each GNN model
# Using ESOL dataset
for model in model_dict.keys():
    for i in grid_search_list:
        RunGNN(esol_data, "ESOL", model, i["h_dim"], i["b_size"], i["lr"])

In [6]:
# Parsing results
dfs = []
for model in model_dict.keys():
    data = open(f"results/GNN/output_{model}_ESOL.txt").readlines()
    temp = []
    for line in data:
        if line[0]!= "P":
            temp.append(float(line.split("=")[1]))
        else:
            temp.append(float(line.split("=")[1].split(",")[0]))
    df = pd.DataFrame(np.array(temp).reshape(48, 6), columns=["LR", "Hidden_Layer", "Batch_Size", "MAE", "RMSE", "r"])
    df["Model"] = model
    dfs.append(df)
final_df = pd.concat(dfs)
final_df.to_csv("results/Output_Hyperparameter_Optimization_GNN.csv", quoting=False, index=False)

In [7]:
# Best parameters
for model in model_dict.keys():
    temp = final_df[final_df["Model"]==model]
    print(temp.sort_values(by=["RMSE"]).head(1),"\n")

       LR  Hidden_Layer  Batch_Size  MAE  RMSE     r      Model
44  0.001         128.0         4.0  0.9  1.13  0.83  SimpleGCN 

       LR  Hidden_Layer  Batch_Size  MAE  RMSE     r      Model
21  0.005          32.0         8.0  0.9  1.12  0.82  SimpleGAT 

       LR  Hidden_Layer  Batch_Size   MAE  RMSE    r      Model
29  0.005         128.0         8.0  0.92  1.17  0.8  SimpleGIN 

       LR  Hidden_Layer  Batch_Size   MAE  RMSE     r            Model
44  0.001         128.0         4.0  0.86  1.07  0.85  SimpleGraphSAGE 

